# Test Renderer

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path (adjust if notebook is in a subfolder)
project_root = Path.cwd().parent  # if notebook is in experiments/ or similar
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


import warnings

# Suppress the specific future warning from torchrl
warnings.filterwarnings(
    "ignore", 
    category=FutureWarning, 
    module="torchrl.modules.mcts.scores"
)

import torch
from torchrl.envs import EnvBase
from torchrl.data import (
    Composite, 
    Unbounded, 
    Bounded,
    Stacked
    
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import numpy as np

# from urbanmarl.envs.urbanmarl_env import UrbanEnv
from urbanmarl.envs.base_env import UrbanEnv
from urbanmarl.envs.rendering import Urban3DRenderer, UrbanRenderConfig

In [ ]:
config = {
    "num_uavs": 3,
    "num_ues": 20,
    "area_size": (500, 500),
    "max_time_slots": 50,
    "max_horizontal_speed": 49.0,
    "max_vertical_speed": 12.0,
    "max_transmit_power": 5.0,
    "frequency_ghz": 29.0,
    "g2a_bandwidth": 10e6,
    "noise_figure_db": 7.0,
    "agents": ["agent_0", "agent_1", "agent_2"]
    # "agents": ["uav_0", "uav_1", 'ue_0']
}
num_envs = 72
scenario = "uav_navigation"
# scenario = "uav_ue_los"
env = UrbanEnv(
    num_envs = num_envs, # batch_size=torch.Size([2])
    continuous_actions = True,
    seed=0,
    device=device,
    scenario=scenario,
    **config)

In [ ]:
# import moviepy as mpy

# img_list = []

# tensordict = env.reset()
# for i in range(100):
#     action = env.action_spec.sample()
#     new_td = tensordict.update(action)
#     next_td = env.step(new_td)
#     img_list.append(env.scenario.render(env))


In [ ]:
n_rollout_steps = 3

tensordict = env.reset()
# rollout = env.rollout(n_rollout_steps)

img = env.scenario.render(env, mode='human')

In [ ]:
action = env.action_spec.sample()
new_td = tensordict.update(action)
next_td = env.step(new_td)
# img_list.append(env.scenario.render(env))
img = env.scenario.render(env, mode='human')

In [ ]:
def render_2d(self, mode: str = "human"):
    import matplotlib.pyplot as plt


    fig, ax = plt.subplots(figsize=(7, 7))
    heat_map = self._env.height_maps[0].cpu().numpy()
    x_min = -float(self._env.volume_size[0]) / 2
    x_max = float(self._env.volume_size[0]) / 2
    y_min = -float(self._env.volume_size[1]) / 2
    y_max = float(self._env.volume_size[1]) / 2

    tpc = ax.imshow(
        heat_map.T,
        cmap="Greys",
        origin="lower",
        extent=[x_min, x_max, y_min, y_max],
        alpha=0.5,
    )
    fig.colorbar(tpc)
    ax.scatter(
        self.ue_user_pos[0, :, 0].cpu(),
        self.ue_user_pos[0, :, 1].cpu(),
        c="blue",
        s=20,
        marker="s",
        label="UEs",
    )
    ax.scatter(
        self.uav_agents_pos[0, :, 0].cpu(),
        self.uav_agents_pos[0, :, 1].cpu(),
        c="red",
        marker="^",
        s=50,
        label="UAVs",
    )
    ax.set_title("UrbanMEC Environment")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.legend(loc="upper right")
    ax.set_aspect("equal")

    if mode == "human":
        plt.show()
    return fig

video_frames = render_2d(env)